# 03 — Comparisons, Error Diagnostics & Operational Action Playbook

**Project:** FlyRank Ranking Signal Analysis — Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 03 of 03 (Model Evaluation, Error Diagnostics & Operational Triage)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
This notebook is your starting template for the final evaluation, diagnostics, and operational actionability phase:
1. **Benchmark Performance Comparisons:** Compile the official evaluation table comparing Baselines vs. Models across $R^2$, RMSE, MAE, Spearman $\rho$, and lift over baseline.
2. **Diagnostic Error Audits:**
   - Visual inspection of Actual vs. Predicted scatter plots and residual distributions.
   - Sub-cohort performance analysis across content types and search intents.
   - In-depth post-mortem audit of 3 concrete outlier cases.
3. **Operational Action Playbook:**
   - Compute the **Opportunity Residual Gap** ($\hat{y} - y$) to detect under-indexed high-potential content.
   - Assign operational triage reason codes (`under_indexed_decay`, `high_demand_opportunity`, `stale_authority`).
   - Export the prioritized refresh queue to `work/outputs/refresh_queue_sample.csv`.

## 1. Official Benchmark Comparisons

All models are evaluated on the standardized, held-out test cohort of **6,163 content items across 7 unseen clients**:
* **Primary Metric:** $R^2$ Score (Coefficient of Determination) — proportion of variance explained on unseen client domains.
* **Secondary Error Metrics:** RMSE and MAE (magnitude of prediction error in natural log units).
* **Monotonic Ranking Quality:** Spearman rank correlation ($\rho$) — measure of sorting accuracy for prioritizing content queues.

In [ ]:
# Code Cell 1: Load Evaluation Payload and Compile Official Benchmark Table
# ----------------------------------------------------------------------
# 1a. Imports & paths
# ----------------------------------------------------------------------
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import r2_score, mean_absolute_error
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y, p): return float(np.sqrt(mean_squared_error(y, p)))

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "work", "outputs", "model_evaluation_payload.pkl")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise FileNotFoundError("model_evaluation_payload.pkl not found — run Notebook 02 first.")
    ROOT = parent
OUT_DIR = os.path.join(ROOT, "work", "outputs")

# ----------------------------------------------------------------------
# 1b. Load payload
# ----------------------------------------------------------------------
with open(os.path.join(OUT_DIR, "model_evaluation_payload.pkl"), "rb") as f:
    payload = pickle.load(f)
X_test = payload["X_test"].reset_index(drop=True)
y_test = payload["y_test"].reset_index(drop=True)
clients_test = payload["clients_test"].reset_index(drop=True)
ids_test = payload["ids_test"].reset_index(drop=True)
preds = {k: np.asarray(v) for k, v in payload["preds"].items()}
MODELS = list(preds)
BASE, CHAMP = MODELS[0], "Champion Stacking Ensemble"
print(f"Test cohort: {len(y_test):,} items across {clients_test.nunique()} unseen clients")

# ----------------------------------------------------------------------
# 1c. Benchmark table
# ----------------------------------------------------------------------
rows = []
for name, p in preds.items():
    rho = spearmanr(y_test, p)[0]
    rows.append({"Model": name, "R2": r2_score(y_test, p), "RMSE": root_mean_squared_error(y_test, p),
                 "MAE": mean_absolute_error(y_test, p), "Spearman_Rho": float(rho) if np.isfinite(rho) else 0.0})
bench = pd.DataFrame(rows)
base_rmse = bench.loc[bench["Model"] == BASE, "RMSE"].iloc[0]
bench["RMSE_Reduction_%"] = 100 * (base_rmse - bench["RMSE"]) / base_rmse

print("\nOFFICIAL BENCHMARK TABLE")
display(bench)

# Markdown version for the report (no tabulate dependency)
hdr = "| " + " | ".join(bench.columns) + " |\n|" + "---|" * len(bench.columns) + "\n"
body = "".join("| " + " | ".join(f"{v:.4f}" if isinstance(v, float) else str(v) for v in r) + " |\n"
               for r in bench.itertuples(index=False))
print(hdr + body)

# ----------------------------------------------------------------------
# 1d. Save metrics JSON
# ----------------------------------------------------------------------
metrics = {r["Model"]: {k: float(v) for k, v in r.items() if k != "Model"} for r in bench.to_dict("records")}
with open(os.path.join(OUT_DIR, "model_benchmark_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved work/outputs/model_benchmark_metrics.json")

In [ ]:
# Code Cell 2: Diagnostic Error Visualizations
y_hat = preds[CHAMP]
resid = y_hat - y_test.values

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

# ----------------------------------------------------------------------
# Subplot 1: Actual vs. Predicted with identity line
# ----------------------------------------------------------------------
axes[0].scatter(y_test, y_hat, s=8, alpha=0.2)
lo, hi = min(y_test.min(), y_hat.min()), max(y_test.max(), y_hat.max())
axes[0].plot([lo, hi], [lo, hi], "r--", lw=2, label="Perfect prediction (y = ŷ)")
axes[0].set(xlabel="Actual log-impressions", ylabel="Predicted log-impressions", title="Actual vs. Predicted (Champion)")
axes[0].legend(loc="upper left")

# ----------------------------------------------------------------------
# Subplot 2: Residual distribution
# ----------------------------------------------------------------------
sns.histplot(resid, bins=60, kde=True, ax=axes[1], color="steelblue")
axes[1].axvline(resid.mean(), color="red", ls="--", label=f"mean = {resid.mean():+.3f}")
axes[1].set(xlabel="Residual (ŷ − y)", title="Residual Distribution")
axes[1].legend()

# ----------------------------------------------------------------------
# Subplot 3: Absolute-error boxplots for every model
# ----------------------------------------------------------------------
abs_err = pd.DataFrame({n: np.abs(p - y_test.values) for n, p in preds.items()}).melt(var_name="Model", value_name="Absolute Error")
sns.boxplot(data=abs_err, x="Model", y="Absolute Error", ax=axes[2], showfliers=False)
axes[2].set_xticklabels([t.get_text().replace(" (", "\n(") for t in axes[2].get_xticklabels()], fontsize=8)
axes[2].set_title("Absolute Error by Model")

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "model_evaluation_plots.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2. Detailed Error Diagnostics & Case Studies

### 2.1 Sub-Cohort Robustness
Audit $R^2$ and MAE across sub-cohorts (`content_type`, `main_intent`) on held-out test domains to ensure the model generalizes across categories rather than overfitting to generic content.

In [ ]:
# Code Cell 3: Sub-Cohort Analysis and Outlier Audits
# ----------------------------------------------------------------------
# 3a. Build one evaluation frame (decode one-hots, back-transform logs to business units)
# ----------------------------------------------------------------------
def decode_onehot(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix)]
    return df[cols].idxmax(axis=1).str.replace(prefix, "", regex=False) if cols else pd.Series("n/a", index=df.index)

ev = X_test.copy()
ev["content_id"], ev["client_id"] = ids_test, clients_test
ev["actual"], ev["pred"] = y_test.values, y_hat
ev["residual"] = ev["pred"] - ev["actual"]
ev["content_type"] = decode_onehot(X_test, "content_type_")
ev["main_intent"] = decode_onehot(X_test, "main_intent_")
for lc, raw in [("log_word_count", "word_count"), ("log_search_volume", "search_volume"),
                ("log_days_since_last_update", "days_since_last_update"),
                ("log_content_age_days", "content_age_days"), ("log_external_backlinks", "external_backlinks")]:
    if lc in ev.columns:
        ev[raw] = np.expm1(ev[lc])

# ----------------------------------------------------------------------
# 3b. Sub-cohort robustness (R2, MAE, Spearman)
# ----------------------------------------------------------------------
def slice_report(col, min_n=100):
    out = []
    for val, g in ev.groupby(col):
        if len(g) < min_n:
            continue
        rho = spearmanr(g["actual"], g["pred"])[0]
        out.append({col: val, "n": len(g), "R2": r2_score(g["actual"], g["pred"]),
                    "MAE": mean_absolute_error(g["actual"], g["pred"]), "Spearman_Rho": rho})
    r = pd.DataFrame(out)
    r["R2_above_0.50"] = r["R2"] > 0.50
    return r

print("By content_type"); display(slice_report("content_type"))
print("By main_intent");  display(slice_report("main_intent"))

per_client = ev.groupby("client_id").apply(lambda g: pd.Series({
    "n": len(g), "R2": r2_score(g["actual"], g["pred"]), "MAE": mean_absolute_error(g["actual"], g["pred"])}))
print("By unseen client"); display(per_client)

# ----------------------------------------------------------------------
# 3c. Three outlier post-mortems
# ----------------------------------------------------------------------
SHOW = [c for c in ["content_id", "client_id", "content_type", "actual", "pred", "residual", "serp_rank_power",
                    "word_count", "search_volume", "days_since_last_update", "external_backlinks"] if c in ev.columns]

print("\n=== CASE 1: Severe UNDER-prediction (ŷ << y) ===")
display(ev.nsmallest(5, "residual")[SHOW])
print("Diagnosis: modest on-page signals but huge realized impressions -> typically a brand/navigational or viral asset "
      "whose demand is invisible to on-page features. Fix: add brand-query / domain-authority features.")

print("\n=== CASE 2: Severe OVER-prediction (ŷ >> y) ===")
display(ev.nlargest(5, "residual")[SHOW])
print("Diagnosis: strong demand + length but near-zero traffic -> head-term cannibalization, an indexing/technical "
      "penalty (Core Web Vitals), or thin affiliate content. Fix: add indexability / duplicate-intent signals.")

strong = (ev["serp_rank_power"] >= ev["serp_rank_power"].quantile(0.75))
if "log_internal_links_in" in ev.columns:
    strong &= ev["log_internal_links_in"] >= ev["log_internal_links_in"].median()
prime = ev[strong & ev["residual"].between(1.5, 3.0)]
print(f"\n=== CASE 3: Prime refresh candidates (strong structure, 1.5 <= ŷ−y <= 3.0): {len(prime):,} pages ===")
display(prime.sort_values("residual", ascending=False).head(5)[SHOW])
print("Diagnosis: healthy SERP power and internal-link equity, yet traffic lags the model's expectation -> "
      "content staleness is the likely lever. Action: high-priority refresh.")

## 3. Conclusions & Operational Action Playbook

### 3.1 Translating Predictions into Business Value
A regression prediction is only useful if it drives operational action. Rather than expecting editorial teams to parse log-impression numbers, construct an **Automated Refresh Triage Queue**:
* **Opportunity Residual Gap ($\text{Residual} = \hat{y} - y$):** 
  * A high positive residual indicates that a page's observed structural depth, semantic properties, and engagement signals warrant substantial search impressions, but its realized traffic is currently lagging far behind expectation.
  * These **under-indexed pages** represent the highest-ROI editorial opportunities: a targeted content refresh can unlock latent ranking potential without starting from scratch.
* **Reason Codes:**
  * `under_indexed_decay`: High predicted potential lagging in realized impressions.
  * `high_demand_opportunity`: High keyword volume and good engagement warranting SERP push.
  * `stale_authority`: Aging high-authority asset at risk of competitive decay.

In [ ]:
# Code Cell 4: Build and Export Operational Refresh Triage Queue
# ----------------------------------------------------------------------
# 4a. Parameters. Template defaults shown; the master guide uses GAP_THRESHOLD=1.2 and top 100 PER CLIENT.
# ----------------------------------------------------------------------
GAP_THRESHOLD = 0.5
TOP_N = 50
PER_CLIENT = False          # True -> take TOP_N per client portfolio instead of overall

# ----------------------------------------------------------------------
# 4b. Opportunity gap + predicted raw impressions
# ----------------------------------------------------------------------
ev["opportunity_gap"] = ev["pred"] - ev["actual"]
ev["predicted_impressions"] = np.expm1(ev["pred"])
ev["actual_impressions"] = np.expm1(ev["actual"])

q = ev[ev["opportunity_gap"] > GAP_THRESHOLD].copy()
print(f"{len(q):,} of {len(ev):,} test pages ({100*len(q)/len(ev):.1f}%) exceed the opportunity-gap threshold {GAP_THRESHOLD}")

# ----------------------------------------------------------------------
# 4c. Triage reason codes (first matching rule wins)
# ----------------------------------------------------------------------
median_pred = ev["pred"].median()
age_col = "days_since_last_update" if "days_since_last_update" in q.columns else "content_age_days"
bl_col = "external_backlinks" if "external_backlinks" in ev.columns else None
bl_q75 = ev[bl_col].quantile(0.75) if bl_col else np.inf
wc_q25 = ev["word_count"].quantile(0.25) if "word_count" in ev.columns else None

conds = [
    (q[age_col] > 180) & (q["pred"] >= median_pred),                                                       # old + high potential
    (q["search_volume"] > 2000) & (q["has_rank_data"] == 1) & q["imputed_avg_position"].between(5, 15),   # demand, stuck mid-SERP
    (q[bl_col] >= bl_q75) if bl_col else pd.Series(False, index=q.index),                                  # authority, lagging traffic
]
codes = ["under_indexed_decay", "high_demand_opportunity", "stale_authority"]
q["triage_reason"] = np.select([c.to_numpy() for c in conds], codes, default="general_gap")

ACTIONS = {
    "under_indexed_decay": "Full refresh: update facts/sections, re-optimize on-page, re-submit for indexing",
    "high_demand_opportunity": "Break into top 5: improve title/H1 & intent match, add internal links, strengthen SERP snippet",
    "stale_authority": "Refresh and consolidate: leverage existing backlink equity, update content and CTAs",
    "general_gap": "Editorial review: audit intent alignment and technical indexing health",
}
q["recommended_action"] = q["triage_reason"].map(ACTIONS)
if wc_q25 is not None:   # word-count-aware tweak for uncategorized pages
    thin = (q["triage_reason"] == "general_gap") & (q["word_count"] < wc_q25)
    q.loc[thin, "recommended_action"] = "Expand thin content: add depth, subtopics and long-tail coverage, then re-index"

q["predicted_visibility"], q["actual_visibility"] = q["pred"], q["actual"]
print("\nQueue composition by reason code:")
display(q.groupby("triage_reason")["opportunity_gap"].agg(["count", "mean"]))

# ----------------------------------------------------------------------
# 4d. Rank, cut and export
# ----------------------------------------------------------------------
q = q.sort_values("opportunity_gap", ascending=False)
queue = q.groupby("client_id", group_keys=False).head(TOP_N) if PER_CLIENT else q.head(TOP_N)

COLS = ["content_id", "client_id", "predicted_visibility", "actual_visibility", "opportunity_gap",
        "predicted_impressions", "actual_impressions", "triage_reason", "recommended_action"]
queue = queue[COLS]
queue.to_csv(os.path.join(OUT_DIR, "refresh_queue_sample.csv"), index=False)
print(f"\nExported {len(queue):,} rows -> work/outputs/refresh_queue_sample.csv")

print("\nTOP 15 PRIORITY CANDIDATES")
display(queue.head(15))

## 4. Executive Summary & Presentation Takeaways
* **Guide for your write-up:**
  * Summarize how your model beat naive heuristics.
  * State the final $R^2$, RMSE reduction, and Spearman rank correlation.
  * Explain how editorial teams should use the refresh queue tomorrow.

In [ ]:
# Executive summary numbers (auto-generated from the results above)
champ = bench.set_index("Model").loc[CHAMP]
base = bench.set_index("Model").loc[BASE]
best_base = bench.iloc[:2].set_index("Model")["RMSE"].min()
print(f"Champion R2             : {champ['R2']:.4f}  (mean baseline R2 = {base['R2']:.4f})")
print(f"RMSE                    : {champ['RMSE']:.4f}  ->  {champ['RMSE_Reduction_%']:.1f}% lower than the mean baseline")
print(f"Spearman rho (ranking)  : {champ['Spearman_Rho']:.4f}")
print(f"Evaluated on            : {clients_test.nunique()} completely unseen clients, {len(y_test):,} pages")
print(f"Refresh queue           : {len(queue):,} pages; top reason = {queue['triage_reason'].mode().iloc[0]}")